# Creating "Statistics of field" from the Meteorological Predictor Fields as Input for RF-based ML-Models
Version 15 December 2022, Selina Kiefer

### Input: csv-files
continuous timeseries of meteorological predictors as 2d-fields in csv-format
### Output: csv-file
continuous timeseries of the minimum, mean, maximum and variance of the meteorological predictors per date in csv-format

#### Define the paths' and files' names 

In [1]:
# Set the needed path and file names.
PATH_defined_functions = './Defined_Functions/'

PATH_input_data = './Data_in_csv_Format/'
ifiles_input_data = ['era5_u10_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                     'era5_z100_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                     'era5_z250_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                     'era5_z500_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                     'era5_z850_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                     'era5_t850_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                     'era5_H850_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                     'era5_u300_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv',
                     'era5_msl_60W_60E_20N_80N_1950_2020_only_Oct_Apr_lead_time_14d.csv']

PATH_output_file = './Predictors_Statistics_of_Field/'
file_name_output_file = 'era5_statistics_u10_z100_z250_z500_z850_t850_H850_u300_msl_60W_60E_20N_80N_1950_2020_lead_time_14d.csv'

#### Import the necessary packages and functions
Nothing needs to be changed here.

In [2]:
# Import the necessary python packages.
import yaml
import pandas as pd
import numpy as np

In [3]:
# Import the necessary functions.
import sys
sys.path.insert(1,PATH_defined_functions)
from read_in_csv_data import *

####  List the predictors to be combined

In [4]:
# List the desired predictors and set how many of these should be taken from the first 
# dataframe. From all other dataframes, only 1 predictor is taken (if more are needed, list
# these input files multiple times in "ifiles_input_data"). It is necessary to take the time as
# a predictor since the data will be grouped by date later.
desired_predictors = ['time', 'month', 'u10', 'z100', 'z250', 'z500', 'z850', 't850', 'H850', 'u300', 'msl']
number_of_predictors_in_first_dataframe = 3
time_column_name = 'time'

#### Combine all predictors into one dataframe
Nothing needs to be changed here.

In [5]:
# A new dataframe is created and the desired predictors from the first data file are written
# into it.
df_combined_input_data = pd.DataFrame()
df_input_data = read_in_csv_data(PATH_input_data, ifiles_input_data[0])
for i in range(number_of_predictors_in_first_dataframe):
    df_combined_input_data[desired_predictors[i]] = df_input_data [desired_predictors[i]]

In [6]:
# From all other dataframes, the specified predictor is added to this new dataframe.
for k in range(len(ifiles_input_data)-1):
    df_input_data = read_in_csv_data(PATH_input_data, ifiles_input_data[k+1])
    df_combined_input_data[desired_predictors[i+k+1]] = df_input_data [desired_predictors[i+k+1]]

#### Calculate the statistics (minimum, mean, maximum and variance) of the predictor fields
Nothing needs to be changed here.

In [7]:
# Now, the time is set as the index and the data is grouped by date. For every desired statistic
# of the field, the calculation is done directly after the grouping and written as separate
# pandas series. Here, the minimum, mean, maximum and variance are calculated.
df_combined_input_data[time_column_name] = pd.to_datetime(df_combined_input_data[time_column_name])
df_combined_input_data = df_combined_input_data.set_index(time_column_name)
ds_input_data_grouped_min = df_combined_input_data.groupby([df_combined_input_data.index.year, df_combined_input_data.index.month, df_combined_input_data.index.day], as_index=False).min()
ds_input_data_grouped_mean = df_combined_input_data.groupby([df_combined_input_data.index.year, df_combined_input_data.index.month, df_combined_input_data.index.day], as_index=False).mean()
ds_input_data_grouped_max = df_combined_input_data.groupby([df_combined_input_data.index.year, df_combined_input_data.index.month, df_combined_input_data.index.day], as_index=False).max()
ds_input_data_grouped_var = df_combined_input_data.groupby([df_combined_input_data.index.year, df_combined_input_data.index.month, df_combined_input_data.index.day], as_index=False).var()

In [8]:
# A new dataframe is created combining all statistics and naming them uniquely.
df_statistics = pd.DataFrame()
for l in range(len(desired_predictors)-1):
    df_statistics['min_'+desired_predictors[l+1]] = ds_input_data_grouped_min[desired_predictors[l+1]]
    df_statistics['mean_'+desired_predictors[l+1]] = ds_input_data_grouped_mean[desired_predictors[l+1]]
    df_statistics['max_'+desired_predictors[l+1]] = ds_input_data_grouped_max[desired_predictors[l+1]]   
    df_statistics['var_'+desired_predictors[l+1]] = ds_input_data_grouped_var[desired_predictors[l+1]]   

In [9]:
# Since the statistics (a single scalar value for each day) of the month are senseless, one of
# the columns is renamed simply with 'month' and the others are removed.
df_statistics = df_statistics.rename(columns={'mean_month':'month'})
df_statistics = df_statistics.drop(['min_month', 'max_month', 'var_month'], axis=1)

#### Add the time information again to the reshaped data
Nothing needs to be changed here.

In [10]:
# Since the time got lost by using .groupby(), a separate new dataframe is created containing
# only the time. To this dataframe, three new columns are added containing the year, month and 
# day.
df_combined_input_data = df_combined_input_data.reset_index()
df_time = pd.DataFrame()
df_time[time_column_name] = pd.to_datetime(df_combined_input_data[time_column_name])
df_time = df_time.set_index(time_column_name)
df_time['year'] = df_time.index.year
df_time['month'] = df_time.index.month
df_time['day'] = df_time.index.day
df_time = df_time.reset_index()

In [11]:
# This new dataframe is then grouped by date and 'averaged' resulting in a daily time-
# series but separated into year, month and day.
df_time = df_time.set_index(time_column_name)
ds_time_mean = df_time.groupby([df_time.index.year, df_time.index.month, df_time.index.day], as_index=False).mean()
df_time_mean = pd.DataFrame(ds_time_mean)

In [12]:
# The separated timeseries is now combined into a single daily timeseries (nothing needs to be
# changed here).
daily_timeseries = (df_time_mean['year'].astype(str)+'-'+df_time_mean['month'].astype(str)+'-'+df_time_mean['day'].astype(str))

In [13]:
# In the next step, the time is added to the dataframe containing the statistics as predictors 
# (nothing needs to be changed here).
df_statistics.insert(0, time_column_name, daily_timeseries)

#### Doublecheck the representation of the data

In [14]:
# Check if everything is reshaped and sorted correctly.
df_statistics.head()

,time,month,min_u10,mean_u10,max_u10,var_u10,min_z100,mean_z100,max_z100,var_z100,...,max_H850,var_H850,min_u300,mean_u300,max_u300,var_u300,min_msl,mean_msl,max_msl,var_msl
0,1950-1-1,1,-12.368885,8.878904,31.195286,72.610363,151657.303091,156672.875435,161518.665356,5.341629e+06,...,111.336243,700.313913,-26.235706,14.941748,60.489780,313.706954,98505.921618,101890.576262,104303.612800,8.651980e+05
1,1950-1-2,1,-10.141999,10.243493,38.581188,74.889176,151490.729802,156454.315335,161125.895230,5.479002e+06,...,106.233137,640.229271,-25.836243,13.824609,66.385611,392.456056,99484.667402,101765.380571,103923.463214,6.346554e+05
2,1950-1-3,1,-8.336727,11.209931,37.465828,82.437378,150888.904610,156216.889052,161056.582855,5.754564e+06,...,117.484758,607.537425,-31.539852,13.751766,63.445204,422.196015,99311.812500,101631.632334,103501.657893,5.857998e+05
3,1950-1-4,1,-8.309897,12.147533,40.440120,112.009832,150771.148102,156056.019105,161007.766075,5.367493e+06,...,126.192910,562.845899,-31.575894,14.121542,60.507800,361.759429,99482.371417,101519.547752,103526.257737,6.951968e+05
4,1950-1-5,1,-10.253152,13.968104,42.900809,152.172610,151178.451467,156016.597060,161459.414454,6.205496e+06,...,107.671922,554.307056,-30.539693,15.099333,72.014132,348.021078,98090.020259,101230.074506,103241.555545,1.039177e+06


In [15]:
# Also check if everything is sorted, renamed or removed correctly at the end of the
# dataframe.
df_statistics.tail()

,time,month,min_u10,mean_u10,max_u10,var_u10,min_z100,mean_z100,max_z100,var_z100,...,max_H850,var_H850,min_u300,mean_u300,max_u300,var_u300,min_msl,mean_msl,max_msl,var_msl
12850,2020-12-13,12,-15.070504,32.003540,75.930778,384.397426,148297.200436,156165.873224,162119.814938,1.192179e+07,...,108.127927,574.475724,-17.635375,15.193222,57.975640,218.461767,96728.075331,101504.287503,104418.948663,1.478467e+06
12851,2020-12-14,12,-16.070564,31.710897,80.369346,392.222339,148345.026966,156060.960809,161846.680613,1.139342e+07,...,110.545742,562.133975,-20.108720,15.354057,56.126846,197.197439,96825.841719,101471.538895,104168.221193,1.383126e+06
12852,2020-12-15,12,-19.143092,31.551365,88.059176,416.061399,148456.746752,156066.828877,161912.442092,1.114977e+07,...,117.466076,582.113599,-24.744688,15.750565,51.860638,185.798974,97398.579494,101392.710143,104138.767522,1.212561e+06
12853,2020-12-16,12,-24.543420,31.242748,96.646932,454.602934,148564.356444,156173.397208,161986.797401,1.185412e+07,...,116.766548,549.574962,-26.177115,15.824722,62.913008,194.945665,95750.906474,101444.625790,104016.002640,1.364801e+06
12854,2020-12-17,12,-33.050321,29.758920,101.847249,583.874074,148611.809330,156150.849015,162515.504746,1.356956e+07,...,118.673596,521.709120,-18.884477,16.115658,65.560357,192.609509,96759.261571,101581.359089,103351.191203,1.211243e+06


#### Save the data in csv format
Nothing needs to be changed here.

In [16]:
# Save the pandas dataframe in csv-format.
df_statistics.to_csv(PATH_output_file+file_name_output_file)

In [17]:
# End of Program